# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.6 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = 'task132'

WORK = Path.cwd()/f'{TASK_ID}_wroking'
WORK.mkdir(parents=True, exist_ok=True)

TASK_JSON = Path(COMPETITION)/'task132.json'
ONNX_PATH = WORK / f'{TASK_ID}.onnx'
SUBMISSION_ZIP = Path.cwd()/'submission.zip'
STATIC_SUBMISSION_ZIP = WORK/'task132_zero_pad_static_graph_submission.zip'
AUDIT_JSON =  WORK/'task132_zero_pad_audit.json'
AUDIT_CSV =  WORK/'task132_zero_pad_audit.csv'
for p in [ONNX_PATH, SUBMISSION_ZIP, STATIC_SUBMISSION_ZIP, AUDIT_JSON, AUDIT_CSV, Path('/mnt/data/submission.zip')]:
    if p.exists():
        p.unlink()
print('WORK:', WORK)

WORK: /kaggle/working/task132_wroking


In [6]:
CH, H, W = 10, 30, 30
FORBIDDEN_OPS = {'Loop','Scan','NonZero','Unique','Script','Function','TreeEnsembleClassifier','TreeEnsembleRegressor'}
RISKY_DYNAMIC_OPS = {'Shape','ConstantOfShape','Expand','Range','ScatterND'}

In [7]:
# Cell 3: task utilities
with open(TASK_JSON, 'r') as f:
    task = json.load(f)


def grid_to_onehot_zero_pad(grid, h=H, w=W, ch=CH):
    """Kaggle/reference contract: real grid is one-hot; padding is all zeros."""
    arr = np.asarray(grid, dtype=np.int64)
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    hh, ww = arr.shape
    for c in range(ch):
        x[0, c, :hh, :ww] = (arr == c).astype(np.float32)
    return x


def expected_tensor_zero_pad(grid, h=H, w=W, ch=CH):
    return grid_to_onehot_zero_pad(grid, h=h, w=w, ch=ch)


def tensor_to_grid_argmax(y, h, w):
    return y[0, :, :h, :w].argmax(axis=0).astype(np.int64)


def expected_bbox_fill(grid):
    arr = np.asarray(grid, dtype=np.int64)
    out = np.zeros_like(arr)
    for color in range(1, 10):
        pos = np.argwhere(arr == color)
        if len(pos) == 0:
            continue
        r0, c0 = pos.min(axis=0)
        r1, c1 = pos.max(axis=0)
        out[r0:r1+1, c0:c1+1] = color
    return out

# Sanity check: the inferred rule matches all labels
for split in ['train', 'test', 'arc-gen']:
    ok = 0
    for ex in task[split]:
        pred = expected_bbox_fill(ex['input'])
        if np.array_equal(pred, np.asarray(ex['output'], dtype=np.int64)):
            ok += 1
    print(split, ok, '/', len(task[split]))
    assert ok == len(task[split])


train 4 / 4
test 1 / 1
arc-gen 262 / 262


In [8]:
# Cell 4: static tensor model
class Task132BBoxFillZeroPad(nn.Module):
    """Static tensor implementation of per-color bounding-box fill.

    Input:  [1,10,30,30] one-hot ARC grid with zero-vector padding.
    Output: [1,10,30,30] one-hot ARC grid with zero-vector padding.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer('lower_h', torch.tril(torch.ones(H, H)).t())
        self.register_buffer('upper_h', torch.triu(torch.ones(H, H)).t())
        self.register_buffer('lower_w', torch.tril(torch.ones(W, W)).t())
        self.register_buffer('upper_w', torch.triu(torch.ones(W, W)).t())
        # Static overlap resolver: highest color wins under accidental overlap.
        self.register_buffer('higher_prior', torch.tril(torch.ones(9, 9), diagonal=-1))

    def forward(self, x):
        # Active real-grid mask. Padding outside the real ARC grid is all-zero in x.
        active = torch.clamp(x.sum(dim=1, keepdim=True), 0.0, 1.0)

        # Foreground channels only: [B,9,H,W]
        m = x[:, 1:10, :, :]

        # Which rows/columns contain each color?
        row_occ = torch.clamp(m.sum(dim=3), 0.0, 1.0)  # [B,9,H]
        col_occ = torch.clamp(m.sum(dim=2), 0.0, 1.0)  # [B,9,W]

        # Static prefix/suffix tests define the interval between first and last marker.
        pref_r = torch.matmul(row_occ, self.lower_h)
        suff_r = torch.matmul(row_occ, self.upper_h)
        pref_c = torch.matmul(col_occ, self.lower_w)
        suff_c = torch.matmul(col_occ, self.upper_w)

        rows_between = (pref_r > 0).float() * (suff_r > 0).float()  # [B,9,H]
        cols_between = (pref_c > 0).float() * (suff_c > 0).float()  # [B,9,W]

        # Cartesian product of row interval and column interval gives the rectangle.
        color_raw = rows_between.unsqueeze(3) * cols_between.unsqueeze(2)  # [B,9,H,W]
        color_raw = color_raw * active

        # Ensure raw one-hot output even under accidental overlaps.
        pix = color_raw.permute(0, 2, 3, 1)  # [B,H,W,9]
        higher_seen = torch.matmul(pix, self.higher_prior)
        pix = pix * (higher_seen <= 0).float()
        color_out = pix.permute(0, 3, 1, 2)

        occupied = torch.clamp(color_out.sum(dim=1, keepdim=True), 0.0, 1.0)
        bg = active * (1.0 - occupied)
        return torch.cat([bg, color_out], dim=1)

model = Task132BBoxFillZeroPad().eval()
print(model)


Task132BBoxFillZeroPad()


In [9]:
# Cell 5: export ONNX
with torch.no_grad():
    # Dummy has a real 15x15 background grid and zero-vector padding outside.
    dummy = torch.zeros((1, CH, H, W), dtype=torch.float32)
    dummy[:, 0, :15, :15] = 1.0
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=['input'],
        output_names=['output'],
        opset_version=13,
        do_constant_folding=True,
        dynamo=False,
    )

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print('ONNX written:', ONNX_PATH)
print('ONNX size:', ONNX_PATH.stat().st_size)


/tmp/ipykernel_16/2153975461.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX written: /kaggle/working/task132_wroking/task132.onnx
ONNX size: 12378


In [10]:
# Cell 6: ONNX audit
onnx_model = onnx.load(str(ONNX_PATH))
ops = {}
for node in onnx_model.graph.node:
    ops[node.op_type] = ops.get(node.op_type, 0) + 1
input_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]
forbidden_found = sorted([op for op in ops if op in FORBIDDEN_OPS])
risky_found = sorted([op for op in ops if op in RISKY_DYNAMIC_OPS])
print('input_shape =', input_shape)
print('output_shape =', output_shape)
print('ops =', ops)
print('forbidden_found =', forbidden_found)
print('risky_found =', risky_found)
assert input_shape == [1, CH, H, W]
assert output_shape == [1, CH, H, W]
assert ONNX_PATH.stat().st_size < 1_400_000
assert not forbidden_found
assert not risky_found


input_shape = [1, 10, 30, 30]
output_shape = [1, 10, 30, 30]
ops = {'Identity': 2, 'Constant': 23, 'ReduceSum': 4, 'Clip': 4, 'Slice': 1, 'MatMul': 5, 'Greater': 4, 'Cast': 5, 'Mul': 6, 'Unsqueeze': 2, 'Transpose': 2, 'LessOrEqual': 1, 'Sub': 1, 'Concat': 1}
forbidden_found = []
risky_found = []


In [11]:
# Cell 7: ONNX Runtime verification on full padded 30x30 tensors
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

def run_onnx(grid):
    x = grid_to_onehot_zero_pad(grid)
    y = sess.run(None, {'input': x})[0]
    return y


def check_examples(examples):
    ok_argmax_real = 0
    ok_raw_tensor = 0
    ok_padding = 0
    for ex in examples:
        inp = np.asarray(ex['input'], dtype=np.int64)
        out = np.asarray(ex['output'], dtype=np.int64)
        h, w = inp.shape
        y = run_onnx(ex['input'])
        pred = tensor_to_grid_argmax(y, h, w)
        if np.array_equal(pred, out):
            ok_argmax_real += 1
        expected_y = expected_tensor_zero_pad(out)
        if np.allclose(y, expected_y, atol=1e-5):
            ok_raw_tensor += 1
        # Padding must be all-zero outside the real input rectangle.
        pad_sum = 0.0
        if h < H:
            pad_sum += float(np.abs(y[:, :, h:, :]).sum())
        if w < W:
            pad_sum += float(np.abs(y[:, :, :, w:]).sum())
        if pad_sum < 1e-5:
            ok_padding += 1
    return ok_argmax_real, ok_raw_tensor, ok_padding, len(examples)

results = {}
for split in ['train', 'test', 'arc-gen']:
    ok, raw_ok, pad_ok, total = check_examples(task[split])
    results[split] = {'ok_argmax_real': ok, 'ok_raw_tensor': raw_ok, 'ok_zero_padding': pad_ok, 'total': total}
    print(split, 'argmax', ok, '/', total, 'raw', raw_ok, '/', total, 'padding', pad_ok, '/', total)

arc = task['arc-gen']
holdout_start = int(len(arc) * 0.40)
holdout = arc[holdout_start:]
ok, raw_ok, pad_ok, total = check_examples(holdout)
results['arc_gen_60pct_holdout'] = {'ok_argmax_real': ok, 'ok_raw_tensor': raw_ok, 'ok_zero_padding': pad_ok, 'total': total}
print('arc_gen_60pct_holdout', 'argmax', ok, '/', total, 'raw', raw_ok, '/', total, 'padding', pad_ok, '/', total)

for k, v in results.items():
    assert v['ok_argmax_real'] == v['total']
    assert v['ok_raw_tensor'] == v['total']
    assert v['ok_zero_padding'] == v['total']


train argmax 4 / 4 raw 4 / 4 padding 4 / 4
test argmax 1 / 1 raw 1 / 1 padding 1 / 1
arc-gen argmax 262 / 262 raw 262 / 262 padding 262 / 262
arc_gen_60pct_holdout argmax 158 / 158 raw 158 / 158 padding 158 / 158


In [12]:
# Cell 8: stronger synthetic generalization validation
# This tests translations, sizes, colors, same-row/same-column cases, and multiple non-overlapping rectangles,
# all under the zero-vector padding contract.

def make_case(h, w, specs):
    inp = np.zeros((h, w), dtype=np.int64)
    out = np.zeros((h, w), dtype=np.int64)
    for color, r0, c0, r1, c1 in specs:
        rr0, rr1 = sorted([r0, r1])
        cc0, cc1 = sorted([c0, c1])
        inp[r0, c0] = color
        inp[r1, c1] = color
        # Avoid overlap in generated multi-object cases; for single-object cases this is exact.
        out[rr0:rr1+1, cc0:cc1+1] = color
    return {'input': inp.tolist(), 'output': out.tolist()}

synthetic = []
for h in [5, 7, 10, 13, 17, 23, 30]:
    for w in [5, 8, 11, 16, 22, 30]:
        for color in [1, 2, 4, 7, 9]:
            synthetic.append(make_case(h, w, [(color, 0, 0, h-1, w-1)]))
            synthetic.append(make_case(h, w, [(color, h//2, 1, h//2, w-2)]))
            synthetic.append(make_case(h, w, [(color, 1, w//2, h-2, w//2)]))
            synthetic.append(make_case(h, w, [(color, 1, 1, h-2, w-2)]))
            synthetic.append(make_case(h, w, [(color, h-2, 1, 1, w-2)]))

# Non-overlapping multi-rectangle cases.
for h, w in [(12, 12), (16, 20), (30, 30), (24, 13)]:
    synthetic.append(make_case(h, w, [
        (2, 0, 0, max(1, h//3-1), max(1, w//3-1)),
        (5, h//2, w//2, h-1, w-1),
    ]))
    synthetic.append(make_case(h, w, [
        (3, 1, 1, h//2-1, w//2-1),
        (8, h//2+1, 1, h-2, w//2-1),
    ]))

ok, raw_ok, pad_ok, total = check_examples(synthetic)
results['synthetic_ood'] = {'ok_argmax_real': ok, 'ok_raw_tensor': raw_ok, 'ok_zero_padding': pad_ok, 'total': total}
print('synthetic_ood', 'argmax', ok, '/', total, 'raw', raw_ok, '/', total, 'padding', pad_ok, '/', total)
assert ok == total
assert raw_ok == total
assert pad_ok == total


synthetic_ood argmax 1058 / 1058 raw 1058 / 1058 padding 1058 / 1058


In [13]:
# Cell 9: write audit files
summary = {
    'task_id': TASK_ID,
    'onnx_path': str(ONNX_PATH),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'input_shape': input_shape,
    'output_shape': output_shape,
    'ops': ops,
    'forbidden_found': forbidden_found,
    'risky_dynamic_found': risky_found,
    'padding_contract': 'zero-vector outside real grid',
    'results': results,
}
with open(AUDIT_JSON, 'w') as f:
    json.dump(summary, f, indent=2)
with open(AUDIT_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['check', 'ok_argmax_real', 'ok_raw_tensor', 'ok_zero_padding', 'total'])
    for k, v in results.items():
        writer.writerow([k, v['ok_argmax_real'], v['ok_raw_tensor'], v['ok_zero_padding'], v['total']])
print(json.dumps(summary, indent=2))


{
  "task_id": "task132",
  "onnx_path": "/kaggle/working/task132_wroking/task132.onnx",
  "onnx_size_bytes": 12378,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Identity": 2,
    "Constant": 23,
    "ReduceSum": 4,
    "Clip": 4,
    "Slice": 1,
    "MatMul": 5,
    "Greater": 4,
    "Cast": 5,
    "Mul": 6,
    "Unsqueeze": 2,
    "Transpose": 2,
    "LessOrEqual": 1,
    "Sub": 1,
    "Concat": 1
  },
  "forbidden_found": [],
  "risky_dynamic_found": [],
  "padding_contract": "zero-vector outside real grid",
  "results": {
    "train": {
      "ok_argmax_real": 4,
      "ok_raw_tensor": 4,
      "ok_zero_padding": 4,
      "total": 4
    },
    "test": {
      "ok_argmax_real": 1,
      "ok_raw_tensor": 1,
      "ok_zero_padding": 1,
      "total": 1
    },
    "arc-gen": {
      "ok_argmax_real": 262,
      "ok_raw_tensor": 262,
      "ok_zero_padding": 262,
      "total": 262
    },
    "arc_gen_60pct_

In [14]:
# Cell 10: write submission zips
for zip_path in [SUBMISSION_ZIP, STATIC_SUBMISSION_ZIP, Path('/mnt/data/submission.zip')]:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        print(zip_path, zf.namelist(), zip_path.stat().st_size)


/kaggle/working/submission.zip ['task132.onnx'] 1502
/kaggle/working/task132_wroking/task132_zero_pad_static_graph_submission.zip ['task132.onnx'] 1502


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/submission.zip'